In [ ]:
# Install dependencies
%pip install -q google-adk google-cloud-aiplatform[adk,agent_engines] litellm requests

In [ ]:
# Restart kernel after installs
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)
print("Runtime restarted. Packages reloaded - continue from here.")

Runtime restarted. Packages reloaded - continue from here.


In [ ]:
# Check Project Variable
import os
print(os.environ.get("GOOGLE_CLOUD_PROJECT"))

qwiklabs-gcp-04-0cd76b3ac58a


In [ ]:
# Init Vertex
import os
import vertexai

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT")
LOCATION = "us-central1"
MAPS_API_KEY = "REDACTED-MAPS-API-KEY"

vertexai.init(project=PROJECT_ID, location=LOCATION)
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

In [ ]:
# Lat/Long Function
import requests
from typing import Optional, Dict, List

def get_lat_long(place: str) -> Optional[Dict[str, float]]:
    """Convert a place name to latitude and longitude using the Google Maps Geocoding API.

    Args:
        place (str): A place name or address, e.g. "Denver, CO".

    Returns:
        Optional[Dict[str, float]]: {"lat": float, "lon": float}, or None if not found.
    """

    url = "https://maps.googleapis.com/maps/api/geocode/json"
    resp = requests.get(url, params={"address": place, "key": MAPS_API_KEY}, timeout=10)
    data = resp.json()

    if data.get("status") != "OK" or not data.get("results"):
        return None

    loc = data["results"][0]["geometry"]["location"]
    print({"lat": loc["lat"], "lon": loc["lng"]})
    return {"lat": loc["lat"], "lon": loc["lng"]}

In [ ]:
# Get the weather forecast
def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """Fetch the extended weather forecast from the US National Weather Service API.

    Args:
        lat (float): Latitude of the location (e.g., 38.8977).
        lon (float): Longitude of the location (e.g., -77.0365).

    Returns:
        Optional[List[Dict[str, str]]]: A list of forecast period dicts,
        or None if data is unavailable or an error occurs.
    """

    headers = {"User-Agent": "adk-weather-agent (workshop@example.com)"}

    try:
        points = requests.get(
            f"https://api.weather.gov/points/{lat},{lon}", headers=headers, timeout=10
        ).json()
        forecast_url = points["properties"]["forecast"]
        periods = requests.get(forecast_url, headers=headers, timeout=10).json()

        return [
            {
                "name": p["name"],
                "temperature": f'{p["temperature"]} {p["temperatureUnit"]}',
                "wind": f'{p["windSpeed"]} {p["windDirection"]}',
                "forecast": p["detailedForecast"],
            }
            for p in periods["properties"]["periods"]
        ]
    except Exception:
        return None

In [ ]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

WEATHER_AGENT_INSTRUCTIONS = """You are Mitch, a friendly US weather assistant.
When a user names a place, call get_lat_long to get coordinates, then call
get_extended_weather_forecast to get the forecast. Summarize current conditions
and call out any alerts (storms, extreme heat/cold, high winds). The NWS API only
covers the United States; if a location is outside the US, say so politely."""

tools = [get_lat_long, get_extended_weather_forecast]
GEMINI_MODEL = "gemini-2.5-flash"

# Gemini version
weather_agent_gemini = Agent(
    name="Gemini_Agent",
    model=GEMINI_MODEL,
    description="Mitch the Friendly Weather Agent (Gemini).",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=tools,
)

# Third-party model version
weather_agent_claude = Agent(
    name="Third_Party_Agent",
    model=LiteLlm(model="MODEL_NAME"),
    description="Mitch the Friendly Weather Agent ({MODEL_NAME}).",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=tools,
)

In [ ]:
# Run the agent across several cities
from vertexai.preview.reasoning_engines import AdkApp
from IPython.display import Markdown, display

def run_agent(agent, cities):
    is_gemini = isinstance(agent.model, str) and agent.model.startswith("gemini")

    if not is_gemini:
        # Only run the Third-Party ONCE for now
        print(f"\n===== {agent.name} =====")
        print("External AI Provider Called")
        return

    app = AdkApp(agent=agent)
    user_id = "test-user-id"
    session = app.create_session(user_id=user_id)
    session_id = session["id"] if isinstance(session, dict) else session.id

    for city in cities:
        print(f"\n===== {agent.name}: {city} =====")
        last = None
        for event in app.stream_query(
            user_id=user_id,
            session_id=session_id,
            message=f"What's the weather and any alerts for {city}?",
        ):
            last = event
        if last is None:
            print("No response (model call failed - see traceback above).")
            continue
        display(Markdown(last["content"]["parts"][0]["text"]))

cities = ["New York, NY", "Denver, CO", "Miami, FL", "Seattle, WA"]
run_agent(weather_agent_gemini, cities)
run_agent(weather_agent_claude, cities)


===== Gemini_Agent: New York, NY =====
{'lat': 40.7127753, 'lon': -74.0059728}


Here's the weather for New York, NY:

**Currently (Today):** Expect a chance of rain showers before 2 PM, followed by showers and thunderstorms, some of which could bring heavy rain. It will be mostly cloudy with a high near 76°F, dropping to around 71°F in the afternoon. Winds will be from the southeast at 5 to 13 mph. There's an 80% chance of precipitation, with new rainfall amounts between three-quarters and one inch possible.

**Tonight:** Showers and thunderstorms are expected, with some potentially producing heavy rain. It will be cloudy with a low around 67°F, rising to 69°F overnight. Winds will be from the south at 6 to 14 mph. There's a 90% chance of precipitation, with new rainfall amounts between 1 and 2 inches possible.

**Alerts:**
*   Be aware of showers and thunderstorms today and tonight, with a possibility of heavy rain.
*   Tonight, rainfall could be significant, with 1 to 2 inches possible.
*   Showers and thunderstorms are also likely on Tuesday.


===== Gemini_Agent: Denver, CO =====
{'lat': 39.7392358, 'lon': -104.990251}


Here's the weather for Denver, CO:

**Currently (Today):** Expect a chance of showers and thunderstorms after noon. It will be mostly sunny with a high near 90°F, dropping to around 88°F in the afternoon. Winds will be from the east-southeast at 2 to 7 mph. There's a 30% chance of precipitation.

**Tonight:** There's a slight chance of showers and thunderstorms before 7 PM. It will be mostly clear with a low around 60°F, rising to 63°F overnight. Winds will be from the south at 3 to 10 mph, with gusts up to 16 mph.

**Alerts:**
*   There's a chance of showers and thunderstorms this afternoon and a slight chance tonight.
*   High temperatures will be near 90°F today.
*   Temperatures will be very high on Saturday, reaching near 98°F.


===== Gemini_Agent: Miami, FL =====
{'lat': 25.7616798, 'lon': -80.1917902}


Here's the weather for Miami, FL:

**Currently (Today):** Expect patchy smoke and sunny skies with a high near 90°F. The heat index could be as high as 105°F. Winds will be from the southeast at 5 to 9 mph.

**Tonight:** There will be patchy smoke before 8 PM, then partly cloudy skies with a low around 83°F. The heat index could be as high as 102°F. Winds will be from the southeast at 5 to 9 mph.

**Alerts:**
*   **Extreme Heat:** Be aware of very high heat index values. It could feel as hot as 105°F today and 102°F tonight.
*   **Smoke:** Patchy smoke is present today and tonight.
*   There's a slight chance of showers and thunderstorms starting Tuesday afternoon and continuing throughout the week.


===== Gemini_Agent: Seattle, WA =====
{'lat': 47.6061389, 'lon': -122.3328481}


Here's the weather for Seattle, WA:

**Currently (Today):** Expect sunny skies with a high near 82°F. Winds will be from the north at 2 to 10 mph.

**Tonight:** It will be partly cloudy with a low around 61°F. Winds will be from the north-northeast at 2 to 9 mph.

**Alerts:**
*   High temperatures are expected, reaching near 82°F today, 86°F on Tuesday, and 87°F on Wednesday.
*   Rain is expected to begin Thursday afternoon and continue through Friday.


===== Third_Party_Agent =====
External AI Provider Called
